In [1]:
import pandas as pd
import spacy
from spellchecker import SpellChecker
import re
import importlib
import utils
import calculate_linguisticFeatures
import numpy as np

importlib.reload(utils)
importlib.reload(calculate_linguisticFeatures)

<module 'calculate_linguisticFeatures' from 'c:\\Projetos\\mineracao_arquivos_epstein_2026\\calculate_linguisticFeatures.py'>

In [65]:
# dt.to_csv('misspellings.csv', sep='|', index=False)

In [15]:
dt_raw = utils.Utils.load('/datasets', 1,2,3,4,5,6,7,8,10,12)

In [34]:
dt_raw['file_type'] = dt_raw['file'].apply(lambda x: x.split('.')[-1])
dt_raw['file_type'] = dt_raw['file_type'].replace({
    'mp4':'video',
    'avi':'video',
    'mov': 'video',
    'm4v': 'video',
    'mp3':'audio',
    'm4a':'audio',
    'wav': 'audio',
    'xlsx': 'sheet',
    'xls': 'sheet',
    'csv': 'sheet',
})
#PDFs sem texto são todos imagens
dt_raw.loc[dt_raw['content'].isna(), 'file_type'] = 'image'

#dropar duplicatas vazias que não são imagens
dt_raw.loc[dt_raw['file_type'] == 'pdf', 'content'] = dt_raw.loc[dt_raw['file_type'] == 'pdf', 'content'].replace({'failed': np.nan})
dt_raw = dt_raw.sort_values('content', na_position='first')
dt_raw = dt_raw.drop_duplicates('file', keep='last')

dt_raw['file_type'].value_counts().to_frame().style

,count
file_type,
pdf,221955
image,8906
video,1226
audio,27
sheet,16
opus,16
amr,5
pluginpayloadattachment,1
3gp,1


In [21]:
dt = dt_raw.loc[dt_raw['file_type'] == 'pdf']
dt = dt.dropna()

In [22]:
linguistics = calculate_linguisticFeatures.linguisticFeatures()

Pré processamento de textos e cálculo de tokens, OOV e orações

In [ ]:
# dt.loc[0, 'preprocessed_text'] = ''
# for i in range(dt.shape[0]):
#   dtemp = linguistics.get(
#       dt['content'].iloc[i],
#       'preprocess',
#   )
#   dt.iloc[i, 5] = dtemp['preprocess'][0]
#   if i%100 == 0:
#     dt.to_csv(f'datasets-preprocess.csv', sep='|', index=False)
# dt.to_csv(f'datasets-preprocess.csv', sep='|', index=False)

In [ ]:
# dt.loc[0, ['count_tokens', 'count_misspellings', 'count_sentences']] = 0
# for i in range(dt.shape[0]):
#   dtemp = linguistics.get(
#       dt['preprocessed_text'].iloc[i],
#       'count_tokens',
#       'count_misspellings',
#       'count_sentences',
#   )
#   dt.iloc[i, [6,7,8]] = dtemp
#   if i%100 == 0:
#     dt.to_csv(f'datasets-misspellings.csv', sep='|', index=False)
# dt.to_csv(f'datasets-misspellings.csv', sep='|', index=False)

In [72]:
dt = pd.read_csv('misspellings.csv', sep='|')

In [67]:
dt['count_tokens'].value_counts()

count_tokens
73.0      2969
50.0      2333
72.0      2323
74.0      2247
68.0      2219
          ... 
5175.0       1
5282.0       1
3466.0       1
7131.0       1
2615.0       1
Name: count, Length: 3371, dtype: int64

In [76]:
dt['ratio_misspelling'] = dt['count_misspellings']/dt['count_tokens']
dt.sort_values('ratio_misspelling', ascending=False)

,dataset,file,content,file_type,len,preprocessed_text,count_tokens,count_misspellings,count_sentences,ratio_misspelling
31,1.0,EFTA00000312.pdf,IL\r\n,pdf,4.0,il,1.0,1.0,1.0,1.0
9,1.0,EFTA00000090.pdf,"7\r\n;ft\r\n7 7 ,\r\n""/\r\n1'11 t>/4\r\n.1\r\n...",pdf,39.0,ft,1.0,1.0,1.0,1.0
59,1.0,EFTA00000490.pdf,.: a -.4 a . oa.\r\n,pdf,18.0,oa,1.0,1.0,1.0,1.0
58,1.0,EFTA00000483.pdf,"At. tt"".\r\n",pdf,10.0,tt,1.0,1.0,1.0,1.0
53,1.0,EFTA00000467.pdf,"it\r\nC.!\r\ntZ\r\n%\r\n-n•Alte\r\nr:12\r\n"" :...",pdf,44.0,tz,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...
131014,10.0,EFTA01763950.pdf,"From:\nSent: Tuesday, July 30, 2013 11:56 AM\n...",pdf,1166.0,send tuesday july jeffrey epstein subject like...,77.0,0.0,3.0,0.0
169597,10.0,EFTA01819302.pdf,To:\nFrom: Jeffrey Epstein\nSent Mon 10/5/2009...,pdf,865.0,jeffrey epstein send mon pm subject mon oct pm...,59.0,0.0,2.0,0.0
5703,8.0,EFTA00027292.pdf,"From:\nTo:'\nSubject: PC.pdf\nDate: Tue, 13 Ap...",pdf,149.0,subject date tue apr attachment case list mont...,12.0,0.0,1.0,0.0
35,1.0,EFTA00000331.pdf,set\r\n,pdf,5.0,set,1.0,0.0,1.0,0.0


In [77]:
dt['ratio_misspelling'].apply(lambda x: np.round(x,1)).value_counts()

ratio_misspelling
0.1    96386
0.0    69183
0.2    35910
0.3    11318
0.4     3898
0.5      952
1.0      534
0.6      445
0.7      249
0.8      159
0.9       27
Name: count, dtype: int64

In [87]:
print('arquivos sem palavras reais: ', (dt['ratio_misspelling'] > 0.2).sum())
print('arquivos com menos de 5 tokens: ', (dt['count_tokens'] < 5).sum())

print(((
    (dt['ratio_misspelling'] > 0.2).sum()
    +(dt['count_tokens'] < 5).sum()
)/dt['preprocessed_text'].size).round(2), '%')

dt_raw.loc[dt_raw['file'].isin(dt.loc[dt['ratio_misspelling'] > 0.2]['file']), ['file_type']] = 'image'

dt = dt.drop(dt.loc[dt['ratio_misspelling'] > 0.2].index)
dt = dt.drop(dt.loc[dt['count_tokens'] < 5].index)
dt = dt.dropna()

arquivos sem palavras reais:  29601
arquivos com menos de 5 tokens:  3163
0.15 %


In [89]:
dt

,dataset,file,content,file_type,len,preprocessed_text,count_tokens,count_misspellings,count_sentences,ratio_misspelling
15,1.0,EFTA00000129.pdf,/III III III III III III\r\n,pdf,26.0,iii iii iii iii iii,5.0,1.0,1.0,0.200000
17,1.0,EFTA00000131.pdf,III III III II III III III III III III\r\n,pdf,40.0,iii iii iii ii iii iii iii iii iii iii,10.0,2.0,1.0,0.200000
171,1.0,EFTA00001188.pdf,PHOTOS OF JE\r\nJE AWARDS\r\nOFFICE\r\nSUPPLIE...,pdf,43.0,photo je je awards office supplies,6.0,1.0,1.0,0.166667
184,1.0,EFTA00001339.pdf,SIAM\r\nA'S\r\ntiv.zthil\r\nCAD\r\nan\r\nGOP'\...,pdf,77.0,siam cad gop ans page mew page door,8.0,1.0,1.0,0.125000
190,1.0,EFTA00001358.pdf,%cards N•ive\r\nCalm Number Sant camp\r\nX ye....,pdf,45.0,card calm number sant camp ye,6.0,1.0,1.0,0.166667
...,...,...,...,...,...,...,...,...,...,...
219056,8.0,EFTA01882596.pdf,From: Office of Terje Rod-Larsen\nSent: Fri 5/...,pdf,46581.0,office terje rod larsen sent fri pm subject up...,3831.0,98.0,18.0,0.025581
219057,8.0,EFTA01882691.pdf,From: Office of Terje Rod-Larsen\nSent: Tue 5/...,pdf,30171.0,office terje rod larsen sent tue pm subject up...,2431.0,96.0,11.0,0.039490
219058,8.0,EFTA01883823.pdf,From: Office of Terje Rod-Larsen\nSent: Thur 8...,pdf,51564.0,office terje rod larsen sent thur pm subject a...,4156.0,103.0,26.0,0.024783
219059,8.0,EFTA01883885.pdf,From: Office of Terje Rod-Larsen\nSent: Mon 5/...,pdf,70762.0,office terje rod larsen sent mon pm subject up...,5621.0,100.0,22.0,0.017790
